Restatement of Main Result

-- paragraph 





---

Restate Casual Declaration 


-- paragraph 





---

In [59]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('../data/clean/merged_stringency_unemployment.csv')

df = df.rename(columns={
    'VIC_Unemployment': 'VIC_unemployment',
    'NSW_Unemployment': 'NSW_unemployment',
    'VIC_Stringency': 'VIC_stringency',
    'NSW_Stringency': 'NSW_stringency'
})

df['Date'] = pd.to_datetime(df['Date'])

In [60]:
vic = df[['Date', 'VIC_unemployment', 'VIC_stringency']].copy()
vic['state'] = 'VIC'
vic = vic.rename(columns={'VIC_unemployment': 'unemployment',
                          'VIC_stringency': 'stringency'})

nsw = df[['Date', 'NSW_unemployment', 'NSW_stringency']].copy()
nsw['state'] = 'NSW'
nsw = nsw.rename(columns={'NSW_unemployment': 'unemployment',
                          'NSW_stringency': 'stringency'})

panel = pd.concat([vic, nsw], ignore_index=True)
panel = panel.dropna(subset=['unemployment', 'stringency']).copy()

In [61]:
panel['treated'] = (panel['state'] == 'VIC').astype(int)
panel['post'] = (panel['Date'] >= '2020-03-01').astype(int)
panel['did'] = panel['treated'] * panel['post']

In [62]:
check1_panel = panel[
    (panel['Date'] >= '2017-01-01') &
    (panel['Date'] <= '2021-12-31')
].copy()

check1_panel['treated'] = (check1_panel['state'] == 'VIC').astype(int)
check1_panel['post'] = (check1_panel['Date'] >= '2020-03-01').astype(int)
check1_panel['did'] = check1_panel['treated'] * check1_panel['post']

model_check1 = smf.ols(
    'unemployment ~ treated + post + did',
    data=check1_panel
).fit(cov_type='HC1')

print("CHECK 1 — Alternative Time Window (2017–2021)")
print(model_check1.summary())

CHECK 1 — Alternative Time Window (2017–2021)
                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.235
Model:                            OLS   Adj. R-squared:                  0.215
Method:                 Least Squares   F-statistic:                     16.05
Date:                Mon, 11 May 2026   Prob (F-statistic):           8.59e-09
Time:                        14:48:19   Log-Likelihood:                -132.30
No. Observations:                 120   AIC:                             272.6
Df Residuals:                     116   BIC:                             283.7
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Interc

In [63]:
placebo_panel = panel.copy()
placebo_panel['post_placebo'] = (placebo_panel['Date'] >= '2018-01-01').astype(int)
placebo_panel['did_placebo'] = placebo_panel['treated'] * placebo_panel['post_placebo']

model_check2 = smf.ols(
    'unemployment ~ treated + post_placebo + did_placebo',
    data=placebo_panel
).fit(cov_type='HC1')

print("CHECK 2 — Placebo Test (Fake Treatment in 2018)")
print(model_check2.summary())

CHECK 2 — Placebo Test (Fake Treatment in 2018)
                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.176
Method:                 Least Squares   F-statistic:                     62.03
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.15e-29
Time:                        14:48:19   Log-Likelihood:                -304.92
No. Observations:                 242   AIC:                             617.8
Df Residuals:                     238   BIC:                             631.8
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------


In [64]:
model_check3 = smf.ols(
    'unemployment ~ treated + post + did',
    data=panel
).fit(cov_type='cluster', cov_kwds={'groups': panel['state']})

print("CHECK 3 — Clustered Standard Errors (State Level)")
print(model_check3.summary())

CHECK 3 — Clustered Standard Errors (State Level)
                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                 1.214e+28
Date:                Mon, 11 May 2026   Prob (F-statistic):           6.42e-15
Time:                        14:48:19   Log-Likelihood:                -309.98
No. Observations:                 242   AIC:                             628.0
Df Residuals:                     238   BIC:                             641.9
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
In

c:\Users\prana\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [65]:
panel['log_unemployment'] = np.log(panel['unemployment'] + 1e-6)

model_check4 = smf.ols(
    'log_unemployment ~ treated + post + did',
    data=panel
).fit(cov_type='HC1')

print("CHECK 4 — Log(Unemployment)")
print(model_check4.summary())

CHECK 4 — Log(Unemployment)
                            OLS Regression Results                            
Dep. Variable:       log_unemployment   R-squared:                       0.178
Model:                            OLS   Adj. R-squared:                  0.168
Method:                 Least Squares   F-statistic:                     21.48
Date:                Mon, 11 May 2026   Prob (F-statistic):           2.40e-12
Time:                        14:48:19   Log-Likelihood:                 71.290
No. Observations:                 242   AIC:                            -134.6
Df Residuals:                     238   BIC:                            -120.6
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.5635   

In [66]:
main_model = smf.ols(
    'unemployment ~ treated + post + did',
    data=panel
).fit(cov_type='HC1')


In [67]:
from statsmodels.iolib.summary2 import summary_col

# Create a neat, well‑ordered robustness table
robust_table = summary_col(
    results=[main_model, model_check1, model_check2, model_check3, model_check4],
    float_format='%0.3f',
    stars=True,
    model_names=[
        'Main Model',
        'Alt Window',
        'Placebo',
        'Clustered SE',
        'Log(Unemp)'
    ],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R²': lambda x: f"{x.rsquared:.3f}"
    },
    regressor_order=[
        'Intercept',
        'treated',
        'post',
        'post_placebo',
        'did',
        'did_placebo'
    ]
)

# Add a clean title and spacing
print("\n" + "="*70)
print("ROBUSTNESS CHECKS — SIDE‑BY‑SIDE COMPARISON")
print("="*70 + "\n")
print(robust_table)
print("\nNote: Standard errors in parentheses. *, **, *** denote significance at 10%, 5%, 1%.")


ROBUSTNESS CHECKS — SIDE‑BY‑SIDE COMPARISON


               Main Model Alt Window  Placebo  Clustered SE Log(Unemp)
----------------------------------------------------------------------
Intercept      4.790***   4.716***   4.957***  4.790***     1.563***  
               (0.055)    (0.062)    (0.073)   (0.000)      (0.011)   
treated        0.581***   0.550***   0.898***  0.581***     0.110***  
               (0.108)    (0.129)    (0.102)   (0.000)      (0.021)   
post           -0.460***  0.933***             -0.460***    -0.125*** 
               (0.137)    (0.197)              (0.000)      (0.029)   
post_placebo                         -0.544***                        
                                     (0.120)                          
did            -0.291     -0.463               -0.291***    -0.040    
               (0.206)    (0.321)              (0.000)      (0.042)   
did_placebo                          -0.606***                        
                              